# Drawing Lines: Linear and Logistic Regression

**CS156 - Finding Patterns in Data with Machine Learning**

This notebook organizes the Session 2 PCW around two simple models on the Iris dataset:

- **linear regression** for predicting a continuous measurement,
- **logistic regression** for binary classification,
- loss functions and evaluation metrics,
- sensitivity / recall,
- implementing both models from scratch.

The notebook keeps the conceptual explanations next to the code so the model-fitting process is visible rather than hidden behind `.fit()`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.datasets import load_iris
from sklearn.linear_model import LinearRegression, LogisticRegression

## 1. Load and inspect the Iris data

The real-world objects are flowers. The columns are measurements such as sepal length, sepal width, petal length, and petal width, while the target label is the flower species.

In [ ]:
def load_iris_data():
    iris = load_iris(as_frame=True)

    df = iris.frame.rename(columns={"target": "species"})
    df["species_name"] = df["species"].map(
        dict(enumerate(iris.target_names))
    )

    return df


df = load_iris_data()
df.head()

## 2. Two different prediction problems

### Linear regression

Use petal length to predict sepal length:

$$
\hat y = \beta_0 + \beta_1 x.
$$

The model selects a line that minimizes squared prediction errors.

### Logistic regression

Use petal length to estimate whether a flower is **virginica** or **not virginica**.

The model begins with a linear score,

$$
z=\beta_0+\beta_1x,
$$

and passes it through the sigmoid:

$$
\sigma(z)=\frac{1}{1+e^{-z}}.
$$

The output lies between 0 and 1 and can be interpreted as an estimated class probability.

In [ ]:
x_col = "petal length (cm)"
y_col = "sepal length (cm)"
positive_species = "virginica"

# Linear regression
X_linear = df[[x_col]]
y_linear = df[y_col]

linear_model = LinearRegression()
linear_model.fit(X_linear, y_linear)

# Logistic regression: virginica vs. rest
X_logistic = df[[x_col]]
y_logistic = (df["species_name"] == positive_species).astype(int)

logistic_model = LogisticRegression()
logistic_model.fit(X_logistic, y_logistic)

print("Linear intercept:", linear_model.intercept_)
print("Linear slope:", linear_model.coef_[0])
print("Logistic classes:", logistic_model.classes_)

### Visualizing the fitted models

The linear model fits a straight line through a cloud of measurements.

The logistic model fits an S-shaped probability curve. It does not try to make every predicted probability exactly 0 or 1; it finds parameters that make the observed class labels probable.

In [ ]:
x_grid = np.linspace(df[x_col].min(), df[x_col].max(), 300).reshape(-1, 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(df[x_col], df[y_col], alpha=0.6)
axes[0].plot(x_grid[:, 0], linear_model.predict(x_grid))
axes[0].set_title("Linear regression")
axes[0].set_xlabel(x_col)
axes[0].set_ylabel(y_col)

axes[1].scatter(df[x_col], y_logistic, alpha=0.35)
axes[1].plot(
    x_grid[:, 0],
    logistic_model.predict_proba(x_grid)[:, 1],
)
axes[1].set_title("Logistic regression: virginica vs. rest")
axes[1].set_xlabel(x_col)
axes[1].set_ylabel("P(virginica)")
axes[1].set_ylim(-0.05, 1.05)

plt.tight_layout()
plt.show()

## 3. What does it mean to "fit" a model?

A model defines a family of possible functions. Training means choosing the parameter values that minimize an objective.

For linear regression, a common objective is mean squared error:

$$
\operatorname{MSE}
=
\frac{1}{n}
\sum_{i=1}^{n}
(y_i-\hat y_i)^2.
$$

For logistic regression, the usual objective is **log loss** / cross-entropy:

$$
-\frac{1}{n}
\sum_i
\left[
y_i\log p_i
+
(1-y_i)\log(1-p_i)
\right].
$$

A model therefore does not need to pass through every training point. It is optimizing a criterion over all observations.

## 4. Evaluating a classifier

Different metrics emphasize different kinds of mistakes.

- **Accuracy:** fraction of all predictions that are correct.
- **Precision:** among predicted positives, the fraction that are truly positive.
- **Sensitivity / recall:** among actual positives, the fraction correctly detected.
- **Specificity:** among actual negatives, the fraction correctly rejected.

For the PCW's virginica-vs-rest model, sensitivity is the relevant example.

In [ ]:
def sensitivity_score(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    true_positives = ((y_true == 1) & (y_pred == 1)).sum()
    false_negatives = ((y_true == 1) & (y_pred == 0)).sum()

    return true_positives / (true_positives + false_negatives)


y_pred = logistic_model.predict(X_logistic)
sensitivity = sensitivity_score(y_logistic, y_pred)

print("Sensitivity:", sensitivity)

## 5. Linear regression from scratch

For one predictor, the least-squares slope has a closed-form solution:

$$
\beta_1
=
\frac{\sum_i (x_i-\bar x)(y_i-\bar y)}
{\sum_i (x_i-\bar x)^2},
$$

and

$$
\beta_0 = \bar y-\beta_1\bar x.
$$

These are exactly the values that minimize the sum of squared residuals for a line with an intercept.

In [ ]:
def linear_regression_from_scratch(
    df,
    x_col="petal length (cm)",
    y_col="sepal length (cm)",
):
    x = df[x_col].to_numpy(dtype=float)
    y = df[y_col].to_numpy(dtype=float)

    x_mean = np.mean(x)
    y_mean = np.mean(y)

    b1 = np.sum((x - x_mean) * (y - y_mean)) / np.sum(
        (x - x_mean) ** 2
    )
    b0 = y_mean - b1 * x_mean

    predictions = b0 + b1 * x
    residuals = y - predictions
    mse = np.mean(residuals ** 2)

    return b0, b1, predictions, mse


linear_b0, linear_b1, linear_predictions, linear_mse = (
    linear_regression_from_scratch(df)
)

print("Intercept:", linear_b0)
print("Slope:", linear_b1)
print("MSE:", linear_mse)

The PCW produced approximately:

- intercept: **4.3066**
- slope: **0.409**
- MSE: **0.1635**

The exact values can vary only if the data or selected columns change.

## 6. Logistic regression from scratch

Logistic regression starts with a linear score and converts it into a probability:

$$
z_i=b_0+b_1x_i,
\qquad
p_i=\sigma(z_i).
$$

For binary cross-entropy, the gradients for this one-feature model can be written as:

$$
\frac{\partial L}{\partial b_0}
=
\frac{1}{n}\sum_i(p_i-y_i)
$$

and

$$
\frac{\partial L}{\partial b_1}
=
\frac{1}{n}\sum_i(p_i-y_i)x_i.
$$

Gradient descent repeatedly moves the parameters in the direction that decreases the loss.

In [ ]:
def logistic_regression_from_scratch(
    df,
    x_col="petal length (cm)",
    positive_species="virginica",
    learning_rate=0.05,
    iterations=20_000,
):
    x = df[x_col].to_numpy(dtype=float)
    y = (df["species_name"] == positive_species).astype(float).to_numpy()

    b0 = 0.0
    b1 = 0.0

    for _ in range(iterations):
        z = b0 + b1 * x
        z = np.clip(z, -500, 500)

        probabilities = 1 / (1 + np.exp(-z))
        error = probabilities - y

        gradient_b0 = np.mean(error)
        gradient_b1 = np.mean(error * x)

        b0 -= learning_rate * gradient_b0
        b1 -= learning_rate * gradient_b1

    z = np.clip(b0 + b1 * x, -500, 500)
    probabilities = 1 / (1 + np.exp(-z))
    predictions = (probabilities >= 0.5).astype(int)

    epsilon = 1e-12
    log_loss = -np.mean(
        y * np.log(probabilities + epsilon)
        + (1 - y) * np.log(1 - probabilities + epsilon)
    )
    accuracy = np.mean(predictions == y)

    return b0, b1, probabilities, predictions, log_loss, accuracy


(
    logistic_b0,
    logistic_b1,
    logistic_probabilities,
    logistic_predictions,
    logistic_loss,
    logistic_accuracy,
) = logistic_regression_from_scratch(df)

print("Intercept:", logistic_b0)
print("Slope:", logistic_b1)
print("Log loss:", logistic_loss)
print("Accuracy:", logistic_accuracy)

## 7. Main takeaways

1. Linear and logistic regression can use the same feature but answer different kinds of questions.
2. A **loss function** defines what the model means by "fit."
3. Linear regression minimizes squared residuals; logistic regression minimizes classification log loss.
4. Metrics such as accuracy, sensitivity, and specificity describe different aspects of classifier performance.
5. Looking inside the optimization makes `.fit()` less mysterious: training means selecting parameter values that reduce an objective.